# STAGE 13B.1 — TEMPORAL EXCLUSION ROOT-CAUSE VERIFICATION
**Malaysia-only FYP | Audit only — no model retraining, no dataset overwriting**

**Purpose:** Investigate the 36 apparently complete cases absent from the full temporal model, the 37 cases with missing values, reference-date evidence and the availability of 1:2 matched controls. The notebook produces review recommendations; it does **not** automatically recover any observation.

**Critical:** A row with non-null predictors is *not automatically eligible*. Event date, historical timing, source quality, matched controls and original Stage-05B selection rules must all be verified.

# PART A — PATHS AND INPUT VALIDATION
## 1. Libraries and project paths

In [1]:
from pathlib import Path
import json
import re
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path(r"C:\Landslide_FYP")
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

STATIC_DIR = DEV_DIR / "03_derived_features" / "static_features_final"
TEMPORAL_DIR = DEV_DIR / "03_derived_features" / "temporal_features_final"
MODEL_DIR = DEV_DIR / "04_model_ready"
PREPROCESS_DIR = DEV_DIR / "01_preprocessing"
STAGE13B_DIR = DEV_DIR / "05_model_results" / "stage13b_record_recovery_audit"
OUTPUT_DIR = DEV_DIR / "05_model_results" / "stage13b1_temporal_root_cause"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATHS = {
    "static": MODEL_DIR / "static_model_1to2.csv",
    "annotated": TEMPORAL_DIR / "temporal_feature_samples_final_annotated.csv",
    "rainfall": TEMPORAL_DIR / "temporal_model_rainfall.csv",
    "rainfall_soilmoisture": TEMPORAL_DIR / "temporal_model_rainfall_soilmoisture.csv",
    "landcover": TEMPORAL_DIR / "temporal_model_landcover.csv",
    "full_enhanced": TEMPORAL_DIR / "temporal_model_full_enhanced.csv",
    "stage13b_candidates": STAGE13B_DIR / "temporal_case_missingness_and_recovery_candidates.csv",
    "stage13b_retention": STAGE13B_DIR / "case_level_retention_matrix.csv",
}

paths_audit = pd.DataFrame([
    {"source": name, "path": str(path), "exists": path.exists()}
    for name, path in PATHS.items()
])
paths_audit.to_csv(OUTPUT_DIR / "input_paths_audit.csv", index=False)
display(paths_audit)

required = ["static", "annotated", "rainfall", "rainfall_soilmoisture", "landcover", "full_enhanced"]
missing_files = [name for name in required if not PATHS[name].exists()]

if missing_files:
    raise FileNotFoundError(f"Missing required Stage 06/05B exports: {missing_files}")

,source,path,exists
0,static,C:\Landslide_FYP\Landslide_FYP_Development\04_model_ready\static_model_1to2.csv,True
1,annotated,C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_feature_samples_fina...,True
2,rainfall,C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_model_rainfall.csv,True
3,rainfall_soilmoisture,C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_model_rainfall_soilm...,True
4,landcover,C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_model_landcover.csv,True
5,full_enhanced,C:\Landslide_FYP\Landslide_FYP_Development\03_derived_features\temporal_features_final\temporal_model_full_enhanced.csv,True
6,stage13b_candidates,C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b_record_recovery_audit\temporal_case_missingness...,True
7,stage13b_retention,C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b_record_recovery_audit\case_level_retention_matr...,True


## 2. Read tables and check label conventions

In [2]:
tables = {
    name: pd.read_csv(path, low_memory=False)
    for name, path in PATHS.items()
    if path.exists()
}


def positive_rows(data):

    if "temporal_role" in data.columns:
        return data.loc[
            data["temporal_role"].astype(str).str.lower().eq("case")
        ].copy()

    if "target" in data.columns:
        return data.loc[
            pd.to_numeric(data["target"], errors="coerce").eq(1)
        ].copy()

    if "sample_class" in data.columns:
        return data.loc[
            data["sample_class"].astype(str).str.lower().eq("landslide")
        ].copy()

    raise KeyError("Cannot determine positive-case rows. Inspect target/sample_class/temporal_role.")


def normalize_ids(series):

    out = series.astype("string").str.strip()
    return out.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})


NAMES = ["static", "annotated", "rainfall", "rainfall_soilmoisture", "landcover", "full_enhanced"]
cases = {}
case_counts = []

for name in NAMES:

    data = positive_rows(tables[name])

    if "sample_id" not in data.columns:
        raise KeyError(f"{name} missing sample_id; cannot perform safe event-level joins")

    data["_case_key"] = normalize_ids(data["sample_id"])

    if data["_case_key"].isna().any() or data["_case_key"].duplicated().any():
        raise ValueError(f"{name} has null/duplicate case IDs. Do not join by coordinates.")

    cases[name] = data
    case_counts.append({"dataset": name, "cases": len(data), "total_rows": len(tables[name])})

counts = pd.DataFrame(case_counts)
counts.to_csv(OUTPUT_DIR / "case_count_consistency.csv", index=False)
display(counts)

EXPECTED = {"static": 194, "rainfall": 189, "rainfall_soilmoisture": 158,
            "landcover": 148, "full_enhanced": 121}

for name, expected in EXPECTED.items():
    if len(cases[name]) != expected:
        print(f"REVIEW: {name} expected {expected}, observed {len(cases[name])}")

,dataset,cases,total_rows
0,static,194,582
1,annotated,189,567
2,rainfall,189,567
3,rainfall_soilmoisture,158,474
4,landcover,148,444
5,full_enhanced,121,363


# PART B — EXCLUSION VS FEATURE-COMPLETE STATUS
## 3. Reconstruct membership across the four temporal datasets

In [3]:
retention = cases["static"][["_case_key"]].copy()

for name in ["rainfall", "rainfall_soilmoisture", "landcover", "full_enhanced"]:
    retention["in_" + name] = retention["_case_key"].isin(set(cases[name]["_case_key"]))

retention["missing_from_full"] = ~retention["in_full_enhanced"]
retention["in_both_components"] = (
    retention["in_rainfall_soilmoisture"]
    & retention["in_landcover"]
)
retention["component_overlap_but_not_full"] = (
    retention["in_both_components"] & retention["missing_from_full"]
)

membership_summary = pd.DataFrame([
    {"group": "Static positives", "cases": len(retention)},
    {"group": "Missing from full temporal", "cases": int(retention["missing_from_full"].sum())},
    {"group": "In both rain+moisture and land-cover subsets", "cases": int(retention["in_both_components"].sum())},
    {"group": "In both components but absent from full", "cases": int(retention["component_overlap_but_not_full"].sum())},
])
retention.to_csv(OUTPUT_DIR / "temporal_membership_crosscheck.csv", index=False)
membership_summary.to_csv(OUTPUT_DIR / "temporal_membership_summary.csv", index=False)
display(membership_summary)

,group,cases
0,Static positives,194
1,Missing from full temporal,73
2,In both rain+moisture and land-cover subsets,121
3,In both components but absent from full,0


## 4. Recheck the 36 complete-feature candidates against the original annotated data

In [4]:
TEMPORAL_FEATURES = [
    "rain_1d_mm",
    "rain_3d_mm",
    "rain_7d_mm",
    "rain_30d_mm",
    "soil_moisture_0_7cm_7d_mean",
    "soil_moisture_7_28cm_7d_mean",
    "landcover_harmonised",
    "distance_waterway_current_m",
]

annotated = cases["annotated"].copy()
missing_columns = [col for col in TEMPORAL_FEATURES if col not in annotated.columns]

if missing_columns:
    raise KeyError(f"Annotated temporal data missing expected columns: {missing_columns}")

feature_flags = annotated[["_case_key"] + TEMPORAL_FEATURES].copy()

for col in TEMPORAL_FEATURES:
    if col != "landcover_harmonised":
        feature_flags[col] = pd.to_numeric(feature_flags[col], errors="coerce")
    else:
        feature_flags[col] = feature_flags[col].replace(r"^\s*$", np.nan, regex=True)

feature_flags["missing_feature_count"] = feature_flags[TEMPORAL_FEATURES].isna().sum(axis=1)
feature_flags["all_temporal_columns_present"] = feature_flags["missing_feature_count"].eq(0)

candidate_audit = retention.merge(
    feature_flags,
    on="_case_key",
    how="left",
    validate="one_to_one",
    indicator="annotated_match",
)

candidate_audit["candidate_complete_but_absent"] = (
    candidate_audit["missing_from_full"]
    & candidate_audit["annotated_match"].eq("both")
    & candidate_audit["all_temporal_columns_present"].fillna(False)
)

candidate_audit["review_reason"] = np.select(
    [
        candidate_audit["annotated_match"].eq("left_only"),
        candidate_audit["candidate_complete_but_absent"],
        candidate_audit["missing_from_full"],
    ],
    [
        "Not present in annotated source",
        "Complete columns: inspect original eligibility and matching rules",
        "Missing temporal predictors: inspect historical-source extraction",
    ],
    default="Already in full temporal subset",
)

candidate_audit.to_csv(OUTPUT_DIR / "case_level_root_cause_candidates.csv", index=False)
complete_cases = candidate_audit.loc[candidate_audit["candidate_complete_but_absent"]].copy()
complete_cases.to_csv(OUTPUT_DIR / "complete_but_excluded_cases.csv", index=False)

print("Missing from full:", int(candidate_audit["missing_from_full"].sum()))
print("Complete but excluded:", len(complete_cases))
display(candidate_audit["review_reason"].value_counts().rename_axis("category").reset_index(name="cases"))

if len(complete_cases) != 36:
    print("REVIEW: Previous Stage 13B identified 36. Investigate why the audit now differs.")

Missing from full: 73
Complete but excluded: 36


,category,cases
0,Already in full temporal subset,121
1,Complete columns: inspect original eligibility and matching rules,36
2,Missing temporal predictors: inspect historical-source extraction,32
3,Not present in annotated source,5


# PART C — HISTORICAL DATE EVIDENCE
## 5. Find candidate event-date and environmental timestamp fields

In [5]:
DATE_PATTERN = re.compile(r"date|time|timestamp|event_day|event_year|year", re.I)

DATESETS = ["static", "annotated", "rainfall", "rainfall_soilmoisture", "landcover", "full_enhanced"]
reported_columns = []

for name in DATESETS:
    for col in tables[name].columns:
        if DATE_PATTERN.search(col):
            reported_columns.append({"dataset": name, "candidate_column": col})

date_columns = pd.DataFrame(reported_columns, columns=["dataset", "candidate_column"])
date_columns.to_csv(OUTPUT_DIR / "available_date_columns.csv", index=False)
display(date_columns)

# Add confirmed original field names here if the extraction notebooks use other labels.
EVENT_DATE_FIELD = None
RAIN_END_FIELD = None
MOISTURE_END_FIELD = None
LANDCOVER_DATE_FIELD = None

# These values stay None until you confirm the relevant Stage 05B definitions.
# The notebook will not guess or fabricate dates.

,dataset,candidate_column
0,annotated,event_date_parsed
1,annotated,event_year
2,annotated,event_day
3,annotated,reference_date
4,annotated,reference_year
5,annotated,landcover_year
6,rainfall,event_date_parsed
7,rainfall,event_year
8,rainfall,event_day
9,rainfall,reference_date


## 6. Check reference-date availability (no leakage claim without timestamp evidence)

In [6]:
def date_series(data, column):

    if column is None or column not in data.columns:
        return pd.Series(pd.NaT, index=data.index, dtype="datetime64[ns, UTC]")

    return pd.to_datetime(data[column], errors="coerce", utc=True)


date_info = annotated[["_case_key"]].copy()
field_map = {
    "event_date": EVENT_DATE_FIELD,
    "rain_window_end": RAIN_END_FIELD,
    "moisture_window_end": MOISTURE_END_FIELD,
    "landcover_reference_date": LANDCOVER_DATE_FIELD,
}

for label, field in field_map.items():
    date_info[label] = date_series(annotated, field)

for label in ["rain_window_end", "moisture_window_end", "landcover_reference_date"]:
    # True only when BOTH dates exist and the environmental reference is strictly before the event.
    date_info[label + "_before_event"] = (
        date_info["event_date"].notna()
        & date_info[label].notna()
        & date_info[label].lt(date_info["event_date"])
    )

date_info["date_evidence_complete"] = date_info[list(field_map)].notna().all(axis=1)
date_info.to_csv(OUTPUT_DIR / "event_and_predictor_date_audit.csv", index=False)

print("Cases with complete source timestamp evidence:", int(date_info["date_evidence_complete"].sum()))
print("A source date that is absent is UNKNOWN, not a failed historical match.")

# Historical annual land-cover should be checked against source calendar conventions.
# A comparison of two timestamps alone cannot prove that an annual map was available at event time.

Cases with complete source timestamp evidence: 0
A source date that is absent is UNKNOWN, not a failed historical match.


# PART D — MATCHED CONTROLS AND FILTER EVIDENCE
## 7. Count eligible control rows per historical case

In [7]:
control_rows = []

for name in ["rainfall", "rainfall_soilmoisture", "landcover", "full_enhanced"]:

    table = tables[name].copy()

    if "temporal_role" not in table.columns or "matched_positive_id" not in table.columns:
        control_rows.append({"dataset": name, "case_key": pd.NA, "controls": np.nan,
                             "note": "Missing temporal_role/matched_positive_id"})
        continue

    is_case = table["temporal_role"].astype(str).str.lower().eq("case")
    is_control = table["temporal_role"].astype(str).str.lower().eq("matched_control")

    case_group = normalize_ids(table.loc[is_case, "matched_positive_id"])
    ctl_group = normalize_ids(table.loc[is_control, "matched_positive_id"])

    if case_group.isna().any() or ctl_group.isna().any():
        print(f"REVIEW: {name} contains null matched_positive_id")

    case_ids = normalize_ids(table.loc[is_case, "sample_id"])
    case_mapping = pd.DataFrame({"case_key": case_ids.values,
                                 "matched_positive_id": case_group.values})

    # Do not assume sample_id == matched_positive_id; verify the explicit mapping.
    if case_mapping["case_key"].duplicated().any():
        raise ValueError(f"{name} duplicate case IDs")

    per_group = ctl_group.value_counts(dropna=False)

    for _, item in case_mapping.iterrows():
        group_id = item["matched_positive_id"]
        controls = int(per_group.get(group_id, 0)) if pd.notna(group_id) else 0
        control_rows.append({"dataset": name, "case_key": item["case_key"],
                             "matched_positive_id": group_id, "controls": controls,
                             "exactly_two": controls == 2})

control_audit = pd.DataFrame(control_rows)
control_audit.to_csv(OUTPUT_DIR / "matched_control_count_audit.csv", index=False)

control_summary = (control_audit.groupby("dataset", dropna=False)
                   .agg(cases=("case_key", "count"),
                        with_two_controls=("exactly_two", "sum"),
                        minimum_controls=("controls", "min"),
                        maximum_controls=("controls", "max"))
                   .reset_index())
control_summary.to_csv(OUTPUT_DIR / "matched_control_summary.csv", index=False)
display(control_summary)

,dataset,cases,with_two_controls,minimum_controls,maximum_controls
0,full_enhanced,121,121,2,2
1,landcover,148,148,2,2
2,rainfall,189,189,2,2
3,rainfall_soilmoisture,158,158,2,2


## 8. Review the original Stage 05B filtering code and record unsupported assumptions

In [8]:
# Source-code inspection is an aid to auditing; it is not automatic proof of why a case was removed.
# Place original 05B notebooks in PREPROCESS_DIR if they are not already there.
notebook_names = [
    "05B_temporal_feature_extraction.ipynb",
    "05B_QC_finalize_temporal_features.ipynb",
    "06_final_pseudoabsence_and_model_dataset (1).ipynb",
    "09_temporal_incremental_benchmark.ipynb",
]

RULE_TERMS = ["dropna", "notna", "isna", "eligible", "valid", "matched_positive_id",
              "landcover_harmonised", "soil_moisture", "rain_", "event_date", "date", "temporal_role"]

rules = []

for filename in notebook_names:
    path = PREPROCESS_DIR / filename
    if not path.exists():
        rules.append({"notebook": filename, "cell": pd.NA, "line": pd.NA,
                      "rule_keyword": "FILE_NOT_FOUND", "source_line": "Inspect manually"})
        continue

    try:
        notebook = json.loads(path.read_text(encoding="utf-8"))
    except Exception as error:
        rules.append({"notebook": filename, "cell": pd.NA, "line": pd.NA,
                      "rule_keyword": "READ_ERROR", "source_line": str(error)})
        continue

    for cell_no, cell in enumerate(notebook.get("cells", [])):
        if cell.get("cell_type") != "code":
            continue
        source = cell.get("source", [])
        source = "".join(source) if isinstance(source, list) else source
        for line_no, line in enumerate(source.splitlines(), 1):
            matched = [term for term in RULE_TERMS if term.lower() in line.lower()]
            if matched:
                rules.append({"notebook": filename, "cell": cell_no, "line": line_no,
                              "rule_keyword": ";".join(matched), "source_line": line.strip()[:300]})

rules_table = pd.DataFrame(rules, columns=["notebook", "cell", "line", "rule_keyword", "source_line"])
rules_table.to_csv(OUTPUT_DIR / "source_filter_rule_inventory.csv", index=False)
display(rules_table.head(40))
print("IMPORTANT: Reconstruct exact eligibility predicates from these source cells before recovery.")

,notebook,cell,line,rule_keyword,source_line
0,05B_temporal_feature_extraction.ipynb,3,15,soil_moisture,RUN_ERA5_SOIL_MOISTURE = True
1,05B_temporal_feature_extraction.ipynb,3,35,eligible,"/ ""malaysia_positive_inventory_ml_eligible.geojson"""
2,05B_temporal_feature_extraction.ipynb,3,64,eligible,"""ML-eligible positive metadata"": POSITIVE_META_PATH,"
3,05B_temporal_feature_extraction.ipynb,5,8,event_date;date,"positive_meta[""event_date_parsed""] = pd.to_datetime("
4,05B_temporal_feature_extraction.ipynb,5,9,event_date;date,"positive_meta.get(""event_date""),"
5,05B_temporal_feature_extraction.ipynb,5,14,event_date;date,"positive_meta[""event_date_parsed""].dt.year"
6,05B_temporal_feature_extraction.ipynb,5,18,event_date;date,"positive_meta[""event_date_parsed""].dt.month"
7,05B_temporal_feature_extraction.ipynb,5,22,event_date;date,"positive_meta[""event_date_parsed""].dt.day"
8,05B_temporal_feature_extraction.ipynb,5,25,date,date_meta = (
9,05B_temporal_feature_extraction.ipynb,5,29,event_date;date,"""event_date_parsed"","


IMPORTANT: Reconstruct exact eligibility predicates from these source cells before recovery.


# PART E — ROOT CAUSE TRIAGE AND DECISION GATE
## 9. Compile case-level triage table

In [9]:
triage = candidate_audit.merge(date_info, on="_case_key", how="left", validate="one_to_one")

# Control availability in the existing full subset does not prove controls can be created
# for excluded cases. Never use unverified or randomly fabricated matches.
full_control = control_audit.loc[
    control_audit["dataset"].eq("full_enhanced"),
    ["case_key", "controls"],
].rename(columns={"case_key": "_case_key", "controls": "existing_full_controls"})
triage = triage.merge(full_control, on="_case_key", how="left", validate="one_to_one")

triage["status"] = np.select(
    [
        ~triage["missing_from_full"],
        triage["candidate_complete_but_absent"],
        triage["missing_from_full"],
    ],
    [
        "RETAINED_ALREADY",
        "REVIEW_SOURCE_FILTERS_AND_MATCHED_CONTROLS",
        "REVIEW_MISSING_SOURCE_FEATURES",
    ],
    default="UNKNOWN",
)

triage["approved_for_retraining"] = False
triage["manual_root_cause"] = ""
triage["manual_evidence_file"] = ""

triage.to_csv(OUTPUT_DIR / "case_level_root_cause_triage.csv", index=False)

decisions = [
    {"gate": "Dataset count and ID audit", "status": "PASS_IF_NO_WARNINGS",
     "detail": "Review printed counts and uniqueness results"},
    {"gate": "Complete-feature excluded case triage", "status": "REVIEW_REQUIRED",
     "detail": f"{len(complete_cases)} apparently complete cases; not approved for recovery"},
    {"gate": "Original Stage 05B eligibility rules", "status": "REVIEW_REQUIRED",
     "detail": "Compare source_filter_rule_inventory.csv with case-level exclusions"},
    {"gate": "Temporal reference dates", "status": "REVIEW_REQUIRED",
     "detail": "Set confirmed field mappings and check extraction windows, including historical annual land-cover"},
    {"gate": "Control matching for added cases", "status": "REVIEW_REQUIRED",
     "detail": "Existing controls for included cases do not establish valid controls for excluded cases"},
    {"gate": "Ready to replace production data", "status": "NOT_APPROVED",
     "detail": "No automatic recovery or retraining was performed"},
]

decision_gate = pd.DataFrame(decisions)
decision_gate.to_csv(OUTPUT_DIR / "stage13b1_decision_gate.csv", index=False)
display(decision_gate)

print("Stage 13B.1 audit outputs saved to:", OUTPUT_DIR)
print("Existing Stage 08–12 benchmarks and production models remain untouched.")

,gate,status,detail
0,Dataset count and ID audit,PASS_IF_NO_WARNINGS,Review printed counts and uniqueness results
1,Complete-feature excluded case triage,REVIEW_REQUIRED,36 apparently complete cases; not approved for recovery
2,Original Stage 05B eligibility rules,REVIEW_REQUIRED,Compare source_filter_rule_inventory.csv with case-level exclusions
3,Temporal reference dates,REVIEW_REQUIRED,"Set confirmed field mappings and check extraction windows, including historical annual land-cover"
4,Control matching for added cases,REVIEW_REQUIRED,Existing controls for included cases do not establish valid controls for excluded cases
5,Ready to replace production data,NOT_APPROVED,No automatic recovery or retraining was performed


Stage 13B.1 audit outputs saved to: C:\Landslide_FYP\Landslide_FYP_Development\05_model_results\stage13b1_temporal_root_cause
Existing Stage 08–12 benchmarks and production models remain untouched.


## Files to send for review

- `case_count_consistency.csv`
- `temporal_membership_summary.csv`
- `complete_but_excluded_cases.csv`
- `case_level_root_cause_triage.csv`
- `available_date_columns.csv`
- `event_and_predictor_date_audit.csv`
- `matched_control_summary.csv`
- `source_filter_rule_inventory.csv`
- `stage13b1_decision_gate.csv`

**Also send the original `05B_temporal_feature_extraction.ipynb` and `05B_QC_finalize_temporal_features.ipynb`** if possible. They are necessary to verify the exact earlier filtering logic, not just infer it from the final CSVs.

**Decision after review:** If additional records pass all original eligibility checks, rebuild the matched temporal dataset, rerun fair LR/RF/XGBoost nested spatial benchmarks, and then refit production models. Otherwise preserve 121 + 242 and advance to national predictor coverage checks.